# YOLO26s on RDD2022-China — fresh, minimal, T4-safe

Self-contained. No custom head-decoding, no fragile plumbing. Every metric comes from ultralytics'
own tested code or a tiny well-scoped helper. Designed to just run on a free T4.

**What it does**
1. Setup + dataset import (Drive zip).
2. Train `yolo26s` 100 epochs (skips if already trained), print official test metrics.
3. Per-class metrics (ultralytics native).
4. Per-image predictions saved once, then a simple **aspect-ratio-bucket recall** analysis in pandas —
   the one number that tells you if thin cracks are the problem.
5. **Head comparison**: NMS-free (default) vs one-to-many+NMS, overall mAP only — memory-safe, one
   model in memory at a time.

Everything writes CSVs to Drive. If a cell fails, the earlier CSVs are already saved.

Classes: `0 D00, 1 D10, 2 D20, 3 D40, 4 repair`.

## 1. Setup

In [ ]:
!pip -q install ultralytics pandas

In [ ]:
import torch, gc, json
from pathlib import Path
import numpy as np, pandas as pd
from ultralytics import YOLO
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f'free {free/1e9:.1f}/{total/1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, zipfile, yaml

# ---------- EDIT THESE IF NEEDED ----------
DRIVE_ZIP  = Path('/content/drive/MyDrive/Research/Dataset/China_Subset.zip')
LOCAL_ROOT = Path('/content/China_Subset')
OUT_DIR    = Path('/content/drive/MyDrive/Research/runs_rdd2022/yolo26s_fresh')
BASE_MODEL = 'yolo26s'
EPOCHS     = 100
IMGSZ      = 640
# ------------------------------------------

CLASS_NAMES = ['longitudinal crack','transverse crack','alligator crack','pothole','repair']
device = 0 if torch.cuda.is_available() else 'cpu'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not LOCAL_ROOT.exists():
    assert DRIVE_ZIP.exists(), f'zip not found: {DRIVE_ZIP}'
    shutil.copy2(DRIVE_ZIP, '/content/China_Subset.zip')
    with zipfile.ZipFile('/content/China_Subset.zip') as z: z.extractall('/content')
    print('extracted')

yaml_path = LOCAL_ROOT / 'dataset.yaml'
cfg = dict(path=str(LOCAL_ROOT), train='train_set/images', val='valid_set/images', nc=5, names=CLASS_NAMES)
if (LOCAL_ROOT/'test_set'/'images').exists(): cfg['test'] = 'test_set/images'
yaml.dump(cfg, open(yaml_path,'w'), sort_keys=False)
print('yaml ready. test split:', cfg.get('test','none'))
for s in ['train_set','valid_set','test_set']:
    d = LOCAL_ROOT/s/'images'
    print(s, len(list(d.glob('*'))) if d.exists() else 'MISSING')

## 2. Train (skips if best.pt exists)

In [ ]:
CKPT = OUT_DIR / 'train' / 'weights' / 'best.pt'
if CKPT.exists():
    print('already trained:', CKPT)
else:
    gc.collect(); torch.cuda.empty_cache()
    YOLO(BASE_MODEL).train(
        data=str(yaml_path), project=str(OUT_DIR), name='train', exist_ok=True,
        epochs=EPOCHS, imgsz=IMGSZ, batch=16, device=device, workers=2,
        optimizer='SGD', lr0=0.01, momentum=0.937, weight_decay=0.0005, warmup_epochs=3,
        mosaic=1.0, patience=0, cache=False, amp=True, plots=False, seed=0)
assert CKPT.exists(), 'training did not produce best.pt'
print('checkpoint:', CKPT)

## 3. Official metrics (ultralytics native) — overall + per class

In [ ]:
gc.collect(); torch.cuda.empty_cache()
m = YOLO(str(CKPT)).val(data=str(yaml_path), split='test', imgsz=IMGSZ, batch=8,
                        device=device, verbose=False)
rows = [dict(klass='ALL', mAP50=round(float(m.box.map50),4), mAP50_95=round(float(m.box.map),4),
             P=round(float(m.box.mp),4), R=round(float(m.box.mr),4))]
for i, name in m.names.items():
    rows.append(dict(klass=name, mAP50=round(float(m.box.ap50[i]),4), mAP50_95=round(float(m.box.maps[i]),4),
                     P='', R=''))
official = pd.DataFrame(rows)
print(official.to_string(index=False))
official.to_csv(OUT_DIR/'official_metrics.csv', index=False)

## 4. Where does it fail? Aspect-ratio bucket recall

Saves per-image predictions once (memory-safe, chunked), then measures **recall per aspect-ratio
bucket** in pandas. If recall collapses for high-AR (thin) boxes, thin cracks are your problem.

In [ ]:
# --- read GT boxes into a flat table ---
def load_gt(split):
    idir, ldir = LOCAL_ROOT/split/'images', LOCAL_ROOT/split/'labels'
    from PIL import Image
    rows = []
    for img in sorted(idir.glob('*')):
        if img.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
        W, H = Image.open(img).size
        lbl = ldir / f'{img.stem}.txt'
        if not lbl.exists(): continue
        for line in lbl.read_text().strip().splitlines():
            p = line.split()
            if len(p) < 5: continue
            c, xc, yc, w, h = int(p[0]), *map(float, p[1:5])
            bw, bh = w*W, h*H
            x1, y1 = xc*W - bw/2, yc*H - bh/2
            rows.append(dict(stem=img.stem, cls=c, x1=x1, y1=y1, x2=x1+bw, y2=y1+bh,
                             ar=max(bw,bh)/max(min(bw,bh),1e-6)))
    return pd.DataFrame(rows)

gt = load_gt('test_set')
gt['ar_bucket'] = pd.cut(gt.ar, [0,2,4,8,1e9], labels=['AR<2','AR2-4','AR4-8','AR>=8'])
print('GT boxes:', len(gt))
print(gt.ar_bucket.value_counts().reindex(['AR<2','AR2-4','AR4-8','AR>=8']).to_string())

In [ ]:
# --- predictions once, chunked, keyed by INPUT path order (ultralytics relabels r.path) ---
def predict_all(model, split, conf=0.25, chunk=64, use_nms_branch=False):
    paths = sorted((LOCAL_ROOT/split/'images').glob('*'))
    paths = [p for p in paths if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
    stems = [p.stem for p in paths]
    rows = []
    with torch.no_grad():
        for i in range(0, len(paths), chunk):
            sp, ss = paths[i:i+chunk], stems[i:i+chunk]
            for stem, r in zip(ss, model.predict([str(p) for p in sp], imgsz=IMGSZ, conf=conf,
                               device=device, verbose=False, stream=True, batch=4)):
                b = r.boxes
                xy = b.xyxy.cpu().numpy(); cf = b.conf.cpu().numpy(); cl = b.cls.cpu().numpy().astype(int)
                for k in range(len(xy)):
                    rows.append(dict(stem=stem, cls=int(cl[k]), conf=float(cf[k]),
                                     x1=float(xy[k,0]), y1=float(xy[k,1]), x2=float(xy[k,2]), y2=float(xy[k,3])))
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)

gc.collect(); torch.cuda.empty_cache()
model = YOLO(str(CKPT))
pred = predict_all(model, 'test_set', conf=0.25)
del model; gc.collect(); torch.cuda.empty_cache()
print('predictions:', len(pred))
pred.to_csv(OUT_DIR/'predictions_test.csv', index=False)

In [ ]:
# --- recall per AR bucket via greedy IoU matching (same class, IoU>=0.5) ---
def iou(a, b):
    x1=np.maximum(a[:,0,None],b[None,:,0]); y1=np.maximum(a[:,1,None],b[None,:,1])
    x2=np.minimum(a[:,2,None],b[None,:,2]); y2=np.minimum(a[:,3,None],b[None,:,3])
    iw=np.clip(x2-x1,0,None); ih=np.clip(y2-y1,0,None); inter=iw*ih
    aa=(a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ab=(b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter/np.clip(aa[:,None]+ab[None,:]-inter,1e-9,None)

gt['matched'] = False
for stem in gt.stem.unique():
    for c in gt[gt.stem==stem].cls.unique():
        g = gt[(gt.stem==stem)&(gt.cls==c)]
        p = pred[(pred.stem==stem)&(pred.cls==c)]
        if len(p)==0: continue
        gb = g[['x1','y1','x2','y2']].values; pb = p[['x1','y1','x2','y2']].values
        M = iou(gb, pb)
        for gi in range(len(gb)):
            if M[gi].max() >= 0.5:
                gt.loc[g.index[gi], 'matched'] = True

rec = gt.groupby('ar_bucket', observed=True).agg(n_gt=('matched','size'),
        recall=('matched','mean')).reindex(['AR<2','AR2-4','AR4-8','AR>=8'])
rec['recall'] = rec['recall'].round(3)
print('Recall @ IoU0.5, conf0.25, by aspect-ratio bucket (test):')
print(rec.to_string())
rec.to_csv(OUT_DIR/'recall_by_ar.csv')

recc = gt.groupby('cls', observed=True).agg(n_gt=('matched','size'), recall=('matched','mean'))
recc.index = [CLASS_NAMES[i] for i in recc.index]
recc['recall'] = recc['recall'].round(3)
print('\nRecall by class:'); print(recc.to_string())
recc.to_csv(OUT_DIR/'recall_by_class.csv')
print('\n>> If recall drops sharply for AR>=8 vs AR<2, thin/elongated cracks are the failure mode.')

## 5. Head comparison: NMS-free vs one-to-many+NMS

The one controlled architectural comparison. Overall test mAP only, one model in memory at a time.
A large drop from o2m to o2o means the NMS-free one-to-one head is costing you performance.

In [ ]:
def quick_map(model, split):
    # ultralytics-native val on the given (already-configured) model
    r = model.val(data=str(yaml_path), split=split, imgsz=IMGSZ, batch=8, device=device, verbose=False)
    return float(r.box.map), float(r.box.map50)

gc.collect(); torch.cuda.empty_cache()

# one-to-one (default, NMS-free)
y = YOLO(str(CKPT))
o2o_map, o2o_map50 = quick_map(y, 'test')
del y; gc.collect(); torch.cuda.empty_cache()

# one-to-many (+NMS): disable end2end
o2m_map = o2m_map50 = None
try:
    y2 = YOLO(str(CKPT))
    y2.model.model[-1].end2end = False
    y2.model.end2end = False
    o2m_map, o2m_map50 = quick_map(y2, 'test')
    del y2; gc.collect(); torch.cuda.empty_cache()
except Exception as e:
    print('one-to-many branch unavailable on this version:', type(e).__name__, str(e)[:150])

head = pd.DataFrame([
    dict(branch='one2one (NMS-free)', mAP50=round(o2o_map50,4), mAP50_95=round(o2o_map,4)),
] + ([dict(branch='one2many (+NMS)', mAP50=round(o2m_map50,4), mAP50_95=round(o2m_map,4))] if o2m_map else []))
print(head.to_string(index=False))
head.to_csv(OUT_DIR/'head_comparison.csv', index=False)
if o2m_map:
    print(f'\no2m - o2o: mAP50-95 {o2m_map - o2o_map:+.4f} | mAP50 {o2m_map50 - o2o_map50:+.4f}')
    print('Positive => the NMS-free one-to-one head is leaving performance on the table.')

## Done

CSVs in `runs_rdd2022/yolo26s_fresh/`: `official_metrics.csv`, `recall_by_ar.csv`,
`recall_by_class.csv`, `head_comparison.csv`, `predictions_test.csv`.

Send those back (especially `recall_by_ar.csv` and `head_comparison.csv`) and we pick the paper's lever.